
# Joint Model
This notebook contains the code for a simple implementation of the Leaspy Joint model on synthetic data.


The following imports are required libraries for numerical computation and data manipulation.



In [8]:
import os

import pandas as pd

import leaspy
from leaspy.io.data import Data

leaspy_root = os.path.dirname(leaspy.__file__)

data_path = os.path.join(leaspy_root, "datasets/data/simulated_data_for_joint.csv")

df = pd.read_csv(data_path, dtype={"ID": str}, sep=";")
print(df.head())

    ID    TIME  EVENT_TIME  EVENT_BOOL       Y0    Y1   Y2   Y3
0  116  78.461        85.5           1  0.44444  0.04  0.0  0.0
1  116  78.936        85.5           1  0.60000  0.00  0.0  0.2
2  116  79.482        85.5           1  0.39267  0.04  0.0  0.2
3  116  79.939        85.5           1  0.58511  0.00  0.0  0.0
4  116  80.491        85.5           1  0.57044  0.00  0.0  0.0


To use the Joint Model in Leaspy, your dataset must include the following columns:

1. **ID** : Patient identifier
2. **TIME** : Time of measurement
3. **EVENT_TIME** : Time of the event
4. **EVENT_BOOL** : Event indicator:
   - `1` if the event occurred
   - `0` if censored
   - `2` if a competing event occurred

For one patient, the event time and event bool are the same for each row.



We load the Joint Model from the `leaspy.models` and transform the dataset in a leaspy-compatible form with the built-in functions.



In [9]:
from leaspy.models import JointModel

data = Data.from_dataframe(df, "joint")
model = JointModel(name="test_model", nb_events=1)

The parameter `nb_events` should match the number of distinct event types
present in the `EVENT_BOOL` column:

- If `EVENT_BOOL` contains values {0, 1}, then `nb_events=1`.
- If it contains values {0, 1, 2}, then `nb_events=2`.

Once the model is initialized, we can fit it to the data.



In [10]:
model.fit(data, "mcmc_saem", seed=1312, n_iter=100, progress_bar=False)

/Users/jv.martini/Documents/leaspy_examples/.venv/lib/python3.9/site-packages/leaspy/models/time_reparametrized.py:288: UserWarning: You did not provide `source_dimension` hyperparameter for multivariate model, setting it to ⌊√dimension⌋ = 2.
  warnings.warn(


 ==> Setting seed to 1312

Fit with `mcmc_saem` took: 4s


The Joint Model includes specific parameters such as `log_rho_mean` and `zeta_mean`.



In [11]:
print(model.parameters)

{'betas_mean': tensor([[-0.0163, -0.0908],
        [-0.0079, -0.0469],
        [-0.1058, -0.0088]]), 'log_g_mean': tensor([0.1157, 2.8874, 2.5624, 1.3001]), 'log_rho_mean': tensor([1.8040]), 'log_v0_mean': tensor([-3.0789, -3.8272, -3.8023, -2.7624]), 'n_log_nu_mean': tensor([-1.9862]), 'noise_std': tensor(0.0947, dtype=torch.float64), 'tau_mean': tensor([78.4523], dtype=torch.float64), 'tau_std': tensor([5.7890], dtype=torch.float64), 'xi_std': tensor([0.4379], dtype=torch.float64), 'zeta_mean': tensor([[0.0421],
        [0.0660]])}
